In [46]:
import pandas as pd
import ast
import re
from collections import Counter

In [47]:
train_df = pd.read_excel("../data/raw/train_fixed.xlsx")
val_df = pd.read_excel("../data/raw/validation_fixed.xlsx")
test_df = pd.read_excel(r"E:\DeepX_Hackathon_neurix\data\raw\unlabeled_fixed.xlsx")

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (1971, 9)
Validation shape: (500, 9)
Test shape: (7047, 7)


In [48]:
train_df.head()

,review_id,review_text,star_rating,date,business_name,business_category,platform,aspects,aspect_sentiments
0,7238,لا يوجد الدفع بالبطاقه عند الاستلام,3,2026-03-08 00:00:00,Noon,ecommerce,play_store,"[""app_experience"", ""delivery""]","{""app_experience"": ""negative"", ""delivery"": ""ne..."
1,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,5,قبل يومين (2),ممشي مصر Mawlana Cafe,كافيه,google_maps,"[""cleanliness"", ""ambiance"", ""service""]","{""cleanliness"": ""positive"", ""ambiance"": ""posit..."
2,1975,تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالول...,1,قبل شهر,بيت لحم Beet Lahm,مطعم,google_maps,"[""service"", ""delivery"", ""food""]","{""service"": ""negative"", ""delivery"": ""negative""..."
3,3024,احلي مكان فزايد,5,قبل شهر,ذا بلكون كافيه الشيخ زايد,مطعم مأكولات ومشروبات,google_maps,"[""general""]","{""general"": ""positive""}"
4,5483,الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا...,4,قبل سنة,The Best Restaurant,مطعم,google_maps,"[""food"", ""price""]","{""food"": ""positive"", ""price"": ""positive""}"


In [49]:
train_df.columns

Index(['review_id', 'review_text', 'star_rating', 'date', 'business_name',
       'business_category', 'platform', 'aspects', 'aspect_sentiments'],
      dtype='object')

In [50]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1971 entries, 0 to 1970
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   review_id          1971 non-null   int64 
 1   review_text        1971 non-null   object
 2   star_rating        1971 non-null   int64 
 3   date               1971 non-null   object
 4   business_name      1971 non-null   object
 5   business_category  1971 non-null   object
 6   platform           1971 non-null   object
 7   aspects            1971 non-null   object
 8   aspect_sentiments  1971 non-null   object
dtypes: int64(2), object(7)
memory usage: 138.7+ KB


In [51]:
val_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   review_id          500 non-null    int64 
 1   review_text        500 non-null    object
 2   star_rating        500 non-null    int64 
 3   date               500 non-null    object
 4   business_name      500 non-null    object
 5   business_category  500 non-null    object
 6   platform           500 non-null    object
 7   aspects            500 non-null    object
 8   aspect_sentiments  500 non-null    object
dtypes: int64(2), object(7)
memory usage: 35.3+ KB


In [52]:

train_df.isnull().sum()


review_id            0
review_text          0
star_rating          0
date                 0
business_name        0
business_category    0
platform             0
aspects              0
aspect_sentiments    0
dtype: int64

In [53]:
for i in range(5):
    print("Review:", train_df.loc[i, "review_text"])
    print("Aspects:", train_df.loc[i, "aspects"])
    print("Sentiments:", train_df.loc[i, "aspect_sentiments"])
    print("-" * 80)

Review: لا يوجد الدفع بالبطاقه عند الاستلام
Aspects: ["app_experience", "delivery"]
Sentiments: {"app_experience": "negative", "delivery": "negative"}
--------------------------------------------------------------------------------
Review: المكان نضيف وجميل وقعدته تحفه والخدمة فوق الممتاز والجو جميل مكان اكتر من رائع بصراحة ❤️❤️❤️❤️
Aspects: ["cleanliness", "ambiance", "service"]
Sentiments: {"cleanliness": "positive", "ambiance": "positive", "service": "positive"}
--------------------------------------------------------------------------------
Review: تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالولي نص ساعة فعد ساعة ونص
الاكل يعني كويس ولكن كمية قليلة جدا
Aspects: ["service", "delivery", "food"]
Sentiments: {"service": "negative", "delivery": "negative", "food": "neutral"}
--------------------------------------------------------------------------------
Review: احلي مكان فزايد
Aspects: ["general"]
Sentiments: {"general": "positive"}
------------------------------------------------------

In [54]:
def parse_list(x):
    if isinstance(x, list):
        return x
    return ast.literal_eval(x)

def parse_dict(x):
    if isinstance(x, dict):
        return x
    return ast.literal_eval(x)

train_df["aspects"] = train_df["aspects"].apply(parse_list)
train_df["aspect_sentiments"] = train_df["aspect_sentiments"].apply(parse_dict)

val_df["aspects"] = val_df["aspects"].apply(parse_list)
val_df["aspect_sentiments"] = val_df["aspect_sentiments"].apply(parse_dict)

In [55]:
train_df[["review_text", "aspects", "aspect_sentiments"]].head()

,review_text,aspects,aspect_sentiments
0,لا يوجد الدفع بالبطاقه عند الاستلام,"[app_experience, delivery]","{'app_experience': 'negative', 'delivery': 'ne..."
1,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,"[cleanliness, ambiance, service]","{'cleanliness': 'positive', 'ambiance': 'posit..."
2,تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالول...,"[service, delivery, food]","{'service': 'negative', 'delivery': 'negative'..."
3,احلي مكان فزايد,[general],{'general': 'positive'}
4,الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا...,"[food, price]","{'food': 'positive', 'price': 'positive'}"


In [56]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = text.strip()

    # شيل URLs
    text = re.sub(r"http\S+|www\S+", " ", text)
    # توحيد الحروف
    text = re.sub(r"[إأآٱ]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)
    
    EMOJI_PATTERN = re.compile(
        "["
        "\U0001F600-\U0001F64F"  
        "\U0001F300-\U0001F5FF"  
        "\U0001F680-\U0001F6FF"  
        "\U0001F1E0-\U0001F1FF"  
        "\U00002700-\U000027BF"  
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE,
    )

    text = re.sub(EMOJI_PATTERN, " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [57]:
train_df["clean_text"] = train_df["review_text"].apply(clean_text)
val_df["clean_text"] = val_df["review_text"].apply(clean_text)
test_df["clean_text"] = test_df["review_text"].apply(clean_text)

In [59]:
def check_row(row):
    return set(row["aspects"]) == set(row["aspect_sentiments"].keys())

train_df["is_valid"] = train_df.apply(check_row, axis=1)
print(train_df["is_valid"].value_counts())

is_valid
True    1971
Name: count, dtype: int64


In [60]:
train_df[["review_text", "clean_text"]].head(20)

,review_text,clean_text
0,لا يوجد الدفع بالبطاقه عند الاستلام,لا يوجد الدفع بالبطاقه عند الاستلام
1,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,المكان نضيف وجميل وقعدته تحفه والخدمه فوق المم...
2,تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالول...,تجربه سييه سالتهم الاكل هياخد وقت قد ايه قالول...
3,احلي مكان فزايد,احلي مكان فزايد
4,الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا...,الفطير حلو جدا الاحجام تحفه بالنسبه للسعر فا ي...
5,لقد حملت تطبيق عن طريق مستر بيست واريد سيارة ت...,لقد حملت تطبيق عن طريق مستر بيست واريد سياره ت...
6,من أجمل المطاعم التي اكلت فيها أكل عربي يمني.\...,من اجمل المطاعم التي اكلت فيها اكل عربي يمني. ...
7,جميلة,جميله
8,"The place is great , clean , service excellent...","The place is great , clean , service excellent..."
9,سى جدا جدا جدا\nوالله والله والله\nمفيش مسؤول ...,سي جدا جدا جدا والله والله والله مفيش مسوول مو...


In [61]:
train_df["original_len"] = train_df["review_text"].astype(str).apply(len)
train_df["clean_len"] = train_df["clean_text"].apply(len)


In [62]:
aspect_counter = Counter()

for aspects in train_df["aspects"]:
    aspect_counter.update(aspects)

aspect_counts = pd.DataFrame(
    aspect_counter.items(),
    columns=["aspect", "count"]
).sort_values(by="count", ascending=False)

aspect_counts

,aspect,count
4,service,988
5,food,454
0,app_experience,453
3,ambiance,378
7,price,354
6,general,303
2,cleanliness,185
1,delivery,161
8,none,57


In [63]:
sentiment_counter = Counter()

for sentiment_dict in train_df["aspect_sentiments"]:
    sentiment_counter.update(sentiment_dict.values())

sentiment_counts = pd.DataFrame(
    sentiment_counter.items(),
    columns=["sentiment", "count"]
).sort_values(by="count", ascending=False)

sentiment_counts

,sentiment,count
1,positive,1646
0,negative,1538
2,neutral,149


In [64]:
def check_row(row):
    aspects = set(row["aspects"])
    sentiment_keys = set(row["aspect_sentiments"].keys())
    return aspects == sentiment_keys

train_df["is_valid_label"] = train_df.apply(check_row, axis=1)

train_df["is_valid_label"].value_counts()

is_valid_label
True    1971
Name: count, dtype: int64

In [65]:
ASPECTS = [
    "food",
    "service",
    "price",
    "cleanliness",
    "delivery",
    "ambiance",
    "app_experience",
    "general",
    "none",
]

for aspect in ASPECTS:
    train_df[f"aspect_{aspect}"] = train_df["aspects"].apply(
        lambda x: 1 if aspect in x else 0
    )

train_df[[f"aspect_{a}" for a in ASPECTS]].head()

,aspect_food,aspect_service,aspect_price,aspect_cleanliness,aspect_delivery,aspect_ambiance,aspect_app_experience,aspect_general,aspect_none
0,0,0,0,0,1,0,1,0,0
1,0,1,0,1,0,1,0,0,0
2,1,1,0,0,1,0,0,0,0
3,0,0,0,0,0,0,0,1,0
4,1,0,1,0,0,0,0,0,0


In [73]:
def build_sentiment_dataset(df):
    rows = []

    for _, row in df.iterrows():
        for aspect in row["aspects"]:
            rows.append({
                "review_id": row["review_id"],
                "text": row["clean_text"],
                "aspect": aspect,
                "input_text": row["clean_text"] + " [ASP] " + aspect,
                "sentiment": row["aspect_sentiments"][aspect]
            })

    return pd.DataFrame(rows)

train_sentiment_df = build_sentiment_dataset(train_df)
val_sentiment_df = build_sentiment_dataset(val_df)

print(train_sentiment_df.head())
print(train_sentiment_df.shape)
print(val_sentiment_df.shape)

train_sentiment_df.to_pickle("../data/processed/train_sentiment.pkl")
val_sentiment_df.to_pickle("../data/processed/validation_sentiment.pkl")

   review_id                                               text  \
0       7238                لا يوجد الدفع بالبطاقه عند الاستلام   
1       7238                لا يوجد الدفع بالبطاقه عند الاستلام   
2       1036  المكان نضيف وجميل وقعدته تحفه والخدمه فوق المم...   
3       1036  المكان نضيف وجميل وقعدته تحفه والخدمه فوق المم...   
4       1036  المكان نضيف وجميل وقعدته تحفه والخدمه فوق المم...   

           aspect                                         input_text sentiment  
0  app_experience  لا يوجد الدفع بالبطاقه عند الاستلام [ASP] app_...  negative  
1        delivery  لا يوجد الدفع بالبطاقه عند الاستلام [ASP] deli...  negative  
2     cleanliness  المكان نضيف وجميل وقعدته تحفه والخدمه فوق المم...  positive  
3        ambiance  المكان نضيف وجميل وقعدته تحفه والخدمه فوق المم...  positive  
4         service  المكان نضيف وجميل وقعدته تحفه والخدمه فوق المم...  positive  
(3333, 5)
(840, 5)


In [38]:
rows = []

for _, row in train_df.iterrows():
    for aspect in row["aspects"]:
        rows.append({
            "text": row["clean_text"],
            "aspect": aspect,
            "input_text": row["clean_text"] + " [ASP] " + aspect,
            "sentiment": row["aspect_sentiments"][aspect]
        })

sentiment_df = pd.DataFrame(rows)

sentiment_df.head()

,text,aspect,input_text,sentiment
0,لا يوجد الدفع بالبطاقه عند الاستلام,app_experience,لا يوجد الدفع بالبطاقه عند الاستلام [ASP] app_...,negative
1,لا يوجد الدفع بالبطاقه عند الاستلام,delivery,لا يوجد الدفع بالبطاقه عند الاستلام [ASP] deli...,negative
2,المكان نضيف وجميل وقعدته تحفه والخدمه فوق المم...,cleanliness,المكان نضيف وجميل وقعدته تحفه والخدمه فوق المم...,positive
3,المكان نضيف وجميل وقعدته تحفه والخدمه فوق المم...,ambiance,المكان نضيف وجميل وقعدته تحفه والخدمه فوق المم...,positive
4,المكان نضيف وجميل وقعدته تحفه والخدمه فوق المم...,service,المكان نضيف وجميل وقعدته تحفه والخدمه فوق المم...,positive


In [67]:
print(train_df.shape)
print(val_df.shape)
print(train_df["clean_text"].isna().sum())
print((train_df["clean_text"] == "").sum())
print(train_df[["review_text", "clean_text", "aspects", "aspect_sentiments"]].head())

(1971, 23)
(500, 10)
0
2
                                         review_text  \
0                لا يوجد الدفع بالبطاقه عند الاستلام   
1  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...   
2  تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالول...   
3                                    احلي مكان فزايد   
4  الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا...   

                                          clean_text  \
0                لا يوجد الدفع بالبطاقه عند الاستلام   
1  المكان نضيف وجميل وقعدته تحفه والخدمه فوق المم...   
2  تجربه سييه سالتهم الاكل هياخد وقت قد ايه قالول...   
3                                    احلي مكان فزايد   
4  الفطير حلو جدا الاحجام تحفه بالنسبه للسعر فا ي...   

                            aspects  \
0        [app_experience, delivery]   
1  [cleanliness, ambiance, service]   
2         [service, delivery, food]   
3                         [general]   
4                     [food, price]   

                                   aspect_sentiments  
0  {'app_ex

In [68]:
def check_row(row):
    return set(row["aspects"]) == set(row["aspect_sentiments"].keys())

train_df["is_valid"] = train_df.apply(check_row, axis=1)
val_df["is_valid"] = val_df.apply(check_row, axis=1)

print(train_df["is_valid"].value_counts())
print(val_df["is_valid"].value_counts())

is_valid
True    1971
Name: count, dtype: int64
is_valid
True    500
Name: count, dtype: int64


In [45]:
print(train_sentiment_df["sentiment"].value_counts())
print(val_sentiment_df["sentiment"].value_counts())


sentiment
positive    1646
negative    1538
neutral      149
Name: count, dtype: int64
sentiment
positive    443
negative    357
neutral      40
Name: count, dtype: int64


In [ ]:
train_df.to_pickle("../data/processed/train_processed.pkl")
val_df.to_pickle("../data/processed/validation_processed.pkl")
test_df.to_pickle("../data/processed/test_processed.pkl")



In [71]:
ASPECTS = [
    "food",
    "service",
    "price",
    "cleanliness",
    "delivery",
    "ambiance",
    "app_experience",
    "general",
    "none",
]

for aspect in ASPECTS:
    train_df[f"aspect_{aspect}"] = train_df["aspects"].apply(lambda x: 1 if aspect in x else 0)
    val_df[f"aspect_{aspect}"] = val_df["aspects"].apply(lambda x: 1 if aspect in x else 0)

In [72]:
train_df.to_pickle("../data/processed/train_processed.pkl")
val_df.to_pickle("../data/processed/validation_processed.pkl")